# E5 · TiderOps · Equipo 04
Evaluación congelada el 29/10/2026. **12 casos sintéticos, 96 ejecuciones**; no son incidentes reales ni un benchmark independiente. Este notebook verifica archivos, recalcula tasas e intervalos, estima el costo y reproduce las respuestas conservadas sin llamar al proveedor.

Ejecutar desde la raíz del repositorio o `eval/e5`, con Python 3.12 y las dependencias de `uv.lock` (`uv sync --extra llm --extra dev --frozen`). Jupyter es opcional: `python eval/course_e5.py --replay` reproduce la evidencia. Las celdas guardadas se ejecutaron secuencialmente con el Python del proyecto; no necesitan bibliotecas de análisis adicionales.

Fuentes: [E5 Canvas](https://utec.instructure.com/courses/25674/assignments/349676), [taller](https://utec.instructure.com/courses/25674/announcements/208885), [repositorio](https://github.com/soyHouston256/tider).

In [1]:
from pathlib import Path
import json, hashlib, statistics, random, math
from collections import defaultdict, Counter
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'eval/e5/manifest.json').exists())
DATA = ROOT/'eval/e5'
manifest = json.loads((DATA/'manifest.json').read_text())
for name, sha in manifest['files'].items():
    assert hashlib.sha256((DATA/name).read_bytes()).hexdigest() == sha, name
cases = json.loads((DATA/'cases.json').read_text())
labels = json.loads((DATA/'labels.json').read_text())
runs = [json.loads(p.read_text()) for p in sorted((DATA/'results/runs').glob('*.json'))]
assert len(cases) == 12 and len(runs) == 96
assert not any('error' in r for r in runs)
for r in runs:
    for name, sha in r['code_sha256'].items():
        assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == sha, name
assert len({(r['case'], r['method'], r['repeat']) for r in runs}) == 96
print('Casos:',len(cases),'| ejecuciones:',len(runs),'| errores API:',0)
print('Métodos:',dict(Counter(r['method'] for r in runs)))
print('Segmentos:',manifest['segments'])
print('Congelación UTC:',manifest['frozen_at_utc'])

Casos: 12 | ejecuciones: 96 | errores API: 0
Métodos: {'agent': 36, 'noop': 12, 'rules': 12, 'simple': 36}
Segmentos: {'explicit': 3, 'variation': 3, 'insufficient': 3, 'stress': 3}
Congelación UTC: 2026-10-29T17:51:20.626925+00:00


## Calidad e incertidumbre
Exactitud: promedio por caso de aciertos / repeticiones. Contrapeso: componentes afirmados incorrectamente / todas las ejecuciones. Abstenerse coincide con la etiqueta únicamente cuando esta es `desconocido`. El bootstrap remuestrea **casos completos**, no respuestas independientes. Semilla 1011; 5.000 remuestreos. Los intervalos describen el corpus sintético, no la calidad en producción.

In [2]:
def percentile(xs, p):
    xs=sorted(xs); idx=(len(xs)-1)*p; lo=math.floor(idx); hi=math.ceil(idx)
    return xs[lo]+(xs[hi]-xs[lo])*(idx-lo)
def interval(values):
    rng=random.Random(1011)
    samples=[statistics.mean(rng.choices(values,k=len(values))) for _ in range(5000)]
    return [percentile(samples,.025),percentile(samples,.975)]
scores={}
summary=json.loads((DATA/'results/summary.json').read_text())
for method in ['noop','rules','simple','agent']:
    rows=[r for r in runs if r['method']==method]
    grouped=defaultdict(list)
    for r in rows:
        expected=labels[r['case']]['expected_component']; predicted=r['result']['component']
        grouped[r['case']].append((predicted==expected,predicted!=expected and predicted!='desconocido'))
    acc=[statistics.mean(v[0] for v in group) for group in grouped.values()]
    wrong=[statistics.mean(v[1] for v in group) for group in grouped.values()]
    scores[method]={c:statistics.mean(v[0] for v in group) for c,group in grouped.items()}
    assert abs(statistics.mean(acc)-summary['methods'][method]['correct']['mean'])<1e-12
    assert interval(acc)==summary['methods'][method]['correct']['ci95']
    print(f'{method:8} n={len(rows):2} exactitud={statistics.mean(acc):.4f} IC95={interval(acc)} error_afirmado={statistics.mean(wrong):.4f}')
for baseline in ['noop','rules','simple']:
    diffs=[scores['agent'][c]-scores[baseline][c] for c in sorted(labels)]
    assert interval(diffs)==summary['paired_deltas'][baseline]['ci95']
    print('Agente menos',baseline,':',round(statistics.mean(diffs)*100,2),'pp; IC95', [round(x*100,2) for x in interval(diffs)])

noop     n=12 exactitud=0.3333 IC95=[0.08333333333333333, 0.5833333333333334] error_afirmado=0.0000
rules    n=12 exactitud=0.6667 IC95=[0.4166666666666667, 0.9166666666666666] error_afirmado=0.0000
simple   n=36 exactitud=0.9722 IC95=[0.9166666666666666, 1.0] error_afirmado=0.0278
agent    n=36 exactitud=0.9167 IC95=[0.75, 1.0] error_afirmado=0.0833
Agente menos noop : 58.33 pp; IC95 [16.67, 91.67]
Agente menos rules : 25.0 pp; IC95 [-8.33, 58.33]
Agente menos simple : -5.56 pp; IC95 [-25.0, 8.33]


In [3]:
for segment in manifest['segments']:
    ids=[c['id'] for c in cases if c['segment']==segment]
    values=[scores['agent'][c] for c in ids]
    print(segment, 'casos=',len(ids),'exactitud=',round(statistics.mean(values),4),'IC95=',interval(values))
print('Errores de los métodos LLM:')
for r in runs:
    if r['method'] in {'simple','agent'} and r['result']['component']!=labels[r['case']]['expected_component']:
        print(r['case'],r['method'],'repetición',r['repeat'],'esperado',labels[r['case']]['expected_component'],'obtenido',r['result']['component'])

explicit casos= 3 exactitud= 1 IC95= [1.0, 1.0]
variation casos= 3 exactitud= 1 IC95= [1.0, 1.0]
insufficient casos= 3 exactitud= 0.6667 IC95= [0.0, 1.0]
stress casos= 3 exactitud= 1 IC95= [1.0, 1.0]
Errores de los métodos LLM:
C09 agent repetición 1 esperado desconocido obtenido frontend
C09 agent repetición 2 esperado desconocido obtenido frontend
C09 agent repetición 3 esperado desconocido obtenido frontend
C10 simple repetición 2 esperado desconocido obtenido redis-cart


## Latencia y costo
La latencia es la observada durante llamadas reales; la duración del replay no la reemplaza. El costo es una estimación de lista con los tokens observados, no una factura. [Tarifa Anthropic consultada el 29/10/2026](https://platform.claude.com/docs/en/about-claude/pricing): Haiku 5.5, US$0,10/M entrada y US$0,50/M salida para prompts hasta 100k, sin caché. Excluye infraestructura y personas.

In [4]:
total=0
for method in ['noop','rules','simple','agent']:
    rows=[r for r in runs if r['method']==method]; costs=[]
    for r in rows:
        cost=0
        for call in r['calls']:
            u=call['usage']
            assert u['input_tokens']<=100000
            assert not u.get('cache_read_input_tokens',0) and not u.get('cache_creation_input_tokens',0)
            cost+=(u['input_tokens']*.10+u['output_tokens']*.50)/1e6
        costs.append(cost)
    times=[r['wall_seconds'] for r in rows]; total+=sum(costs)
    print(method,'US$ medio / mín / máx:',[round(x,7) for x in [statistics.mean(costs),min(costs),max(costs)]],
          'segundos mín / máx / p95:',[round(x,3) for x in [min(times),max(times),percentile(times,.95)]])
assert abs(total-json.loads((DATA/'results/costs.json').read_text())['total_usd'])<1e-12
print('Total estimado US$:',round(total,7))

noop US$ medio / mín / máx: [0, 0, 0] segundos mín / máx / p95: [0.0, 0.0, 0.0]
rules US$ medio / mín / máx: [0, 0, 0] segundos mín / máx / p95: [0.0, 0.0, 0.0]
simple US$ medio / mín / máx: [0.0002996, 0.0001359, 0.0005589] segundos mín / máx / p95: [1.497, 5.02, 4.444]
agent US$ medio / mín / máx: [0.0031307, 0.0016481, 0.0075988] segundos mín / máx / p95: [15.537, 58.343, 52.038]
Total estimado US$: 0.1234876


## Reproducción con red bloqueada
Se vuelven a ejecutar el grafo y las herramientas de lectura, utilizando las respuestas guardadas. Cada solicitud debe coincidir con su hash; solo se normaliza el tiempo transcurrido. Una llamada nueva al LLM puede variar. El replay no ejecuta reparaciones ni certifica validez causal.

In [5]:
import importlib.util, socket
from unittest.mock import patch
spec=importlib.util.spec_from_file_location('e5_course',ROOT/'eval/course_e5.py')
module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
checked=0
with patch('socket.create_connection',side_effect=AssertionError('Network disabled')), patch.object(socket.socket,'connect',side_effect=AssertionError('Network disabled')):
    for r in runs:
        case=next(c for c in cases if c['id']==r['case'])
        model=module.RecordedModel(r['model'],replay=r['calls'])
        result=module.perform(case,r['method'],model)
        assert module.signature(result)==module.signature(r['result'])
        assert len(model.records)==len(r['calls'])
        checked+=1
assert checked==96
print('Replay: 96/96 coinciden; sockets bloqueados durante la comprobación.')

Replay: 96/96 coinciden; sockets bloqueados durante la comprobación.


## Decisión y límites
El agente cumple los umbrales puntuales preregistrados para la demo asistida (exactitud ≥80%, errores afirmados ≤10%, p95 ≤120 s, cero acciones y replay coincidente). Su IC de exactitud comienza en 75%: no demuestra que la calidad poblacional supere 80%. **No supera al LLM simple en esta muestra** y consume más tiempo y tokens. C09 muestra atribución injustificada al frontend; los cambios se probarán en un conjunto nuevo.

Pendientes: revisión semántica y de citas por anotadores independientes, κ, tres usuarios elegibles y al menos uno real, mejora basada en ese estudio, acceso empresarial autorizado y alcance con asesor. No se declara reducción de MTTR, ahorro humano ni recuperación. Max: código/reproducción; José: evaluación/errores; Alberto: datos/usuarios.